# Getting Started

ProbeDataTools is designed to make common electron-microprobe calculations
transparent, reproducible, and easy to inspect.

This guide walks through the basic workflow:

1. Load analytical data into a pandas DataFrame.
2. Define the analysed species.
3. Create a `ProbeData` object.
4. Inspect the associated stoichiometric information.
5. Calculate atoms per formula unit (APFU).
6. Check the resulting cation total.

The examples use a small amphibole dataset, but the same workflow applies
to other minerals and liquids.

In [41]:
import pandas as pd

from probedatatools import ProbeData
from probedatatools.cations import calc_apfu, calc_cat_tot

## Example Data

In this example, we create a DataFrame manually. Analytical data can also be read directly from an Excel spreadsheet, provided that the analytical columns use recognised species names, e.g. SiO2, Cr2O3, F, and Cl.

ProbeDataTools functions are vectorised and can operate on any number of analyses. Each row represents a single analysis, while columns contain analytical species and any associated metadata.

In [42]:
amph = pd.DataFrame({
    "SiO2":  [41.773],
    "TiO2":  [ 1.421],
    "Al2O3": [11.644],
    "FeO":   [13.960],
    "MgO":   [12.132],
    "CaO":   [ 9.414],
    "Na2O":  [ 2.375],
    "K2O":   [ 0.410],
    "Cl":    [ 0.078],
}, index=["Example Amphibole"])

display(amph.T)

,Example Amphibole
SiO2,41.773
TiO2,1.421
Al2O3,11.644
FeO,13.960
MgO,12.132
CaO,9.414
Na2O,2.375
K2O,0.410
Cl,0.078


## Defining the analytical species

`ProbeData` needs to know which columns contain analytical species. The order does not determine the calculation; their  names are used to look up the corresponding stoichiometric information.

In [43]:
species = ["SiO2", "TiO2", "Al2O3", "FeO", "MgO", "CaO", "Na2O", "K2O", "Cl",]

amph_pd = ProbeData(data=amph, species=species)

`ProbeData` associates each analytical species with the information required for subsequent calculations, including its molar mass, cation and oxygen stoichiometry, ionic charge, and anion stoichiometry.

For example the molar masses used for this dataset can be inspected with:

In [44]:
amph_pd.MR_use

SiO2      60.08
TiO2      79.88
Al2O3    101.96
FeO       71.85
MgO       40.30
CaO       56.08
Na2O      61.98
K2O       94.20
Cl        35.45
Name: MR, dtype: float64

The stoichiometric information is read from `species.csv`, which currently contains definitions for 46 common analytical species. Additional species can be added to this file without modifying the calculation functions.

This separates the analytical species from the calculations themselves. As a result, the same functions can be applied to datasets containing different combinations of oxides and other analytical species without requiring a separate calculation workflow for each analytical setup.

## Calculating atoms per formula unit

`calc_apfu()` normalises the analysed composition to a specified number of
oxygens per formula unit.

For amphibole, we use a 23-oxygen basis.

In [45]:
apfu = calc_apfu(probe_data=amph_pd, afu=23.0)

display(apfu.T)

,Example Amphibole
Si,6.462225
Ti,0.165338
Al,2.122847
Fe2,1.805822
Mg,2.797974
Ca,1.560208
Na,0.712293
K,0.080906
Cl,0.020450


When total Fe is supplied as FeO, it is initially represented as Fe2 in the APFU calculation. Fe²⁺/Fe³⁺ partitioning, where required, is performed separately using the Fe-partitioning functions.

### Non-oxygen anions

Analysed non-oxygen anions such as F and Cl are recognised automatically from the `ProbeData` species definitions.

They are calculated using the same oxygen renormalisation factor as the cations, but they do not contribute to the oxygen normalisation itself.

Thus, Cl appears in the APFU result above, but the oxygen basis remains 23 O.

## Checking the cation total

The `calc_cat_tot()` function adds a `cat_tot` column containing only the
analysed cations. Non-oxygen anions such as Cl are excluded from this total.

In [46]:
formula = calc_cat_tot(probe_data=amph_pd, afu=23)

display(formula.T)

,Example Amphibole
Si,6.462225
Ti,0.165338
Al,2.122847
Fe2,1.805822
Mg,2.797974
Ca,1.560208
Na,0.712293
K,0.080906
Cl,0.020450
cat_tot,15.707613


## Where to go next

`calc_apfu()` does not need to know the mineral identity. The user supplies the appropriate oxygen basis, while ProbeData supplies the stoichiometric information for the analysed species.

For example:

- olivine is commonly calculated on a 4-O basis;
- pyroxene on a 6-O basis;
- amphibole on a 23-O basis.

The next sections of the documentation cover mineral-specific calculations,including Fe²⁺/Fe³⁺ partitioning and endmember calculations.